# A1 — 조건부 staged routing (노션 A-4 수행 + C 기반 강도 분리)

**질문:** 한 번의 forward 안에서 L7(early)·L13(late) 판단으로 행동과 강도를 고르면, Always-retain(동결 설정)보다 나은가?

- A-4 ① L7 routing(intervention gain 기준) · ② L13 판단(K, C, gain) · ③ No / early-only / late-only / staged 비교 · ④ 지표 전부 포함
- A0 대비 추가: **강한 retain(ρ0.09, 0.12)** — C−로 판단된 행에만 강하게 걸어 C+를 지키는 routing 고유 이득 검증
- **GPU·생성 없음.** 행동 결과는 A0 결과표 + 현경 exp2 파일, router 판단은 저장된 h_out(개입 전과 동일)으로 재현
- 한 행에는 early 또는 late 중 하나만 개입 → 결과표 조회가 실제 생성과 같음 (greedy)

| 구분 | 행동 (arm) | 출처 |
|---|---|---|
| no-op | base | A0 |
| early (L8–12) | early_ret_r0.03/0.06, early_acc_r0.03/0.06 | A0 |
| late (L14–18) | late_ret_r0.03 / 0.06 / **0.09 / 0.12**, late_acc_r0.03/0.06 | A0 + steer_v3 exp2 |

**평가:** 5-fold 교차 적합 + fold 내부 nested CV로 임계값·행동 매핑 선택 (test fold는 선택에 쓰지 않음). 선택 시 inner C+ ΔEM ≥ −1%p 후보를 우선(안전 제약). 질문 단위 bootstrap CI, McNemar.
**실행:** CPU 런타임으로 충분. 약 10~20분.

In [1]:
# ===== A1-0. 환경·경로 =====
import os, json, time, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from google.colab import drive; drive.mount("/content/drive")
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from scipy.stats import binomtest

ROOT  = "/content/drive/MyDrive/11주차_A축_실험(ldy)/"
P9    = ROOT + "P9/"                    # 공유 폴더 바로가기 (읽기 전용)
SV3   = P9 + "steer_v3/"
A0    = ROOT + "A0_headroom/"
OUT   = ROOT + "A1_staged_routing/"; os.makedirs(OUT, exist_ok=True)

SEED, PCA_DIM, N_BOOT, N_PERM = 0, 128, 2000, 200
L_EARLY, L_LATE = 7, 13                 # 판독층 (0-based). 개입은 각각 L8–12, L14–18
SAFETY = -0.02                          # 사전등록 안전 기준: C+ ΔEM 95% CI 하한 > −2%p
SEL_CP_MIN = -0.01                      # 파라미터 선택 시 inner C+ ΔEM 하한 (넘는 후보 우선, 없으면 EM 최대)
for f in [A0 + "gen_base.parquet", SV3 + "exp2_P9_r0.09.parquet", P9 + "hidden_v2/p8s_v2_h_out.npy",
          SV3 + "env/row_manifest.parquet", SV3 + "cb_v3.parquet"]:
    assert os.path.exists(f), f"없음: {f}"
print("ok")

Mounted at /content/drive
ok


In [3]:
# ===== A1-1. 결과표·라벨·hidden (모든 arm = A0 직접 생성본) =====
m   = pd.read_csv(P9 + "hidden_v2/p8s_behavior_v2.csv")
man = pd.read_parquet(SV3 + "env/row_manifest.parquet")
assert (man.question.values == m.question.values).all()
N = len(m); fold = man.fold.values.astype(int)
cb = pd.read_parquet(SV3 + "cb_v3.parquet")
yK = (m.question.map(dict(zip(cb.question, cb.kside_v3))).str.replace("−", "-") == "K+").astype(int).values
yC = (m.ctx_label.str.replace("−", "-") == "C+").astype(int).values          # 1 = C+
isCm = yC == 0
CELL = np.array([("K+" if k else "K-") + ("C+" if c else "C-") for k, c in zip(yK, yC)])

SRC = {"base": A0 + "gen_base.parquet",
       "early_ret_r0.03": A0 + "gen_early_ret_r0.03.parquet", "early_ret_r0.06": A0 + "gen_early_ret_r0.06.parquet",
       "early_acc_r0.03": A0 + "gen_early_acc_r0.03.parquet", "early_acc_r0.06": A0 + "gen_early_acc_r0.06.parquet",
       "late_ret_r0.03": A0 + "gen_late_ret_r0.03.parquet", "late_ret_r0.06": A0 + "gen_late_ret_r0.06.parquet",
       "late_ret_r0.09": A0 + "gen_late_ret_r0.09.parquet", "late_ret_r0.12": A0 + "gen_late_ret_r0.12.parquet",
       "late_acc_r0.03": A0 + "gen_late_acc_r0.03.parquet", "late_acc_r0.06": A0 + "gen_late_acc_r0.06.parquet"}
for a, f in SRC.items():
    assert f.startswith(A0) and os.path.exists(f), f"직접 생성본 없음: {a} → A0 노트북 A0-5b 먼저 실행"
print("모든 arm 출처: A0_headroom (직접 생성)")

ACT = list(SRC); EARLY = [a for a in ACT if a.startswith("early")]; LATE = [a for a in ACT if a.startswith("late")]
E, INFO = {}, []
for a, f in SRC.items():
    d = pd.read_parquet(f); assert (d.rid.values == np.arange(N)).all(), f
    E[a] = d.em.values.astype(float)
    INFO.append(dict(arm=a, C_minus=E[a][isCm].mean(), C_plus=E[a][~isCm].mean(), hit=d.hit_limit.mean(), rep3=d.rep3.mean()))
E = np.stack([E[a] for a in ACT], 1); base = E[:, 0]; REF = ACT.index("late_ret_r0.06")
G = E - base[:, None]                                                          # 행×행동 개입 이득
print(pd.DataFrame(INFO).round(4).to_string(index=False))
assert abs(E[:, REF].mean() - 0.5997) < 1e-3, "A0 Always-retain과 불일치"

# 참고: 현경 exp2와 대조 (분석에는 사용하지 않음)
for r in ["0.03", "0.09", "0.12"]:
    mine, hk = pd.read_parquet(SRC[f"late_ret_r{r}"]), pd.read_parquet(SV3 + f"exp2_P9_r{r}.parquet")
    print(f"late_ret_r{r}: 현경 exp2 대비 문자열 불일치 {int((mine.raw.values != hk.raw.values).sum())}/{N} | "
          f"EM {mine.em.mean():.4f} vs {hk.em.mean():.4f}")

H = np.load(P9 + "hidden_v2/p8s_v2_h_out.npy", mmap_mode="r")
X7, X13 = np.asarray(H[:, L_EARLY]).astype(np.float32), np.asarray(H[:, L_LATE]).astype(np.float32)
q = m.question.values; uq = np.unique(q); qidx = {u: np.where(q == u)[0] for u in uq}
rs_ = np.random.RandomState(SEED); BOOT = [np.concatenate([qidx[u] for u in rs_.choice(uq, len(uq))]) for _ in range(N_BOOT)]
print("hidden", X7.shape, "| 셀:", pd.Series(CELL).value_counts().to_dict())

모든 arm 출처: A0_headroom (직접 생성)
            arm  C_minus  C_plus    hit   rep3
           base   0.2598  0.8023 0.0000 0.0000
early_ret_r0.03   0.3072  0.8023 0.0000 0.0000
early_ret_r0.06   0.3529  0.8056 0.0000 0.0000
early_acc_r0.03   0.2157  0.8023 0.0000 0.0000
early_acc_r0.06   0.1569  0.8105 0.0000 0.0000
 late_ret_r0.03   0.3431  0.8023 0.0000 0.0000
 late_ret_r0.06   0.3954  0.8039 0.0000 0.0000
 late_ret_r0.09   0.4265  0.7745 0.0000 0.0000
 late_ret_r0.12   0.4216  0.7320 0.0016 0.0025
 late_acc_r0.03   0.1699  0.7908 0.0000 0.0000
 late_acc_r0.06   0.0948  0.7843 0.0000 0.0000
late_ret_r0.03: 현경 exp2 대비 문자열 불일치 0/1224 | EM 0.5727 vs 0.5727
late_ret_r0.09: 현경 exp2 대비 문자열 불일치 0/1224 | EM 0.6005 vs 0.6005
late_ret_r0.12: 현경 exp2 대비 문자열 불일치 0/1224 | EM 0.5768 vs 0.5768
hidden (1224, 4096) | 셀: {'K+C+': 346, 'K+C-': 346, 'K-C+': 266, 'K-C-': 266}


In [4]:
# ===== A1-2. 판독 모델 (학습 행에서만 적합) =====
def fit_predict(tr, te):
    # tr 행으로 적합, te 행에 대한 예측 반환
    P = {}
    for tag, X in [("7", X7), ("13", X13)]:
        sc = StandardScaler().fit(X[tr]); pca = PCA(PCA_DIM, random_state=SEED).fit(sc.transform(X[tr]))
        Ztr, Zte = pca.transform(sc.transform(X[tr])), pca.transform(sc.transform(X[te]))
        zs = StandardScaler().fit(Ztr); Ztr, Zte = zs.transform(Ztr), zs.transform(Zte)
        P["pK" + tag] = LogisticRegression(C=0.1, max_iter=3000).fit(Ztr, yK[tr]).predict_proba(Zte)[:, 1]
        if tag == "13":
            P["pC13"] = LogisticRegression(C=0.1, max_iter=3000).fit(Ztr, yC[tr]).predict_proba(Zte)[:, 1]
        acts = EARLY if tag == "7" else LATE
        idx = [ACT.index(a) for a in acts]
        P["g" + tag] = Ridge(alpha=100.0).fit(Ztr, G[tr][:, idx]).predict(Zte)    # 행동별 예상 이득
    return P

def oof_predict(rows, n_split=4, seed=SEED):
    # rows 안에서 질문 단위 inner CV로 OOF 예측 (nested 선택용)
    uq_ = np.unique(q[rows]); rng = np.random.RandomState(seed); rng.shuffle(uq_)
    grp = {u: i % n_split for i, u in enumerate(uq_)}; g = np.array([grp[x] for x in q[rows]])
    out = None
    for j in range(n_split):
        tr, te = rows[g != j], rows[g == j]
        Pj = fit_predict(tr, te)
        if out is None: out = {k: np.zeros((len(rows),) + v.shape[1:]) for k, v in Pj.items()}
        for k, v in Pj.items(): out[k][g == j] = v
    return out

In [5]:
# ===== A1-3. 정책 정의 — 반환값은 행별 행동 인덱스 (ACT 기준) =====
iE, iL = np.array([ACT.index(a) for a in EARLY]), np.array([ACT.index(a) for a in LATE])

def pol_gain(P, key, idxs, margin):
    # 예상 이득이 가장 큰 행동, 그 값이 margin 미만이면 no-op
    g = P[key]; best = g.argmax(1); a = idxs[best].copy(); a[g.max(1) < margin] = 0; return a

def state_of(P, tK, tC, use_K):
    c = (P["pC13"] >= tC).astype(int)
    return (c + 2 * (P["pK13"] >= tK).astype(int)) if use_K else c

def learn_map(states, rows, cand):
    # 예측 상태별로 (학습 행 기준) 평균 EM이 가장 높은 행동
    return {s: cand[int(np.argmax(E[rows][states == s][:, cand].mean(0)))] if (states == s).any() else REF
            for s in np.unique(states)}

GRID = {
    "early_gain":  [dict(margin=mg) for mg in [-9, 0.0, 0.02, 0.05, 0.1]],
    "late_gain":   [dict(margin=mg) for mg in [-9, 0.0, 0.02, 0.05, 0.1]],
    "late_stateC": [dict(tC=t) for t in [0.3, 0.5, 0.7]],
    "late_stateKC":[dict(tC=t, tK=u) for t in [0.3, 0.5, 0.7] for u in [0.3, 0.5, 0.7]],
    "staged_gain": [dict(te=te, margin=mg) for te in [0.02, 0.05, 0.1, 0.2, 9] for mg in [-9, 0.0, 0.05]],
    "staged_state":[dict(te=te, tC=t, tK=u) for te in [0.02, 0.05, 0.1, 0.2, 9] for t in [0.3, 0.5, 0.7] for u in [0.5]],
}
LATE_CAND = np.array([0] + list(iL))                                          # late 단계 후보: no-op + late 행동

def apply(name, P, prm, mapping=None):
    if name == "early_gain": return pol_gain(P, "g7", iE, prm["margin"])
    if name == "late_gain":  return pol_gain(P, "g13", iL, prm["margin"])
    if name in ("late_stateC", "late_stateKC", "staged_state"):
        s = state_of(P, prm.get("tK", .5), prm["tC"], name != "late_stateC")
        a = np.array([mapping.get(x, REF) for x in s])
        if name == "staged_state":
            e = pol_gain(P, "g7", iE, prm["te"]); a = np.where(e > 0, e, a)
        return a
    if name == "staged_gain":
        e = pol_gain(P, "g7", iE, prm["te"]); l = pol_gain(P, "g13", iL, prm["margin"]); return np.where(e > 0, e, l)

def fit_policy(name, P_in, rows):
    # nested: rows의 inner-OOF 예측 P_in으로 파라미터·매핑 선택
    best = None
    for prm in GRID[name]:
        mp = None
        if name in ("late_stateC", "late_stateKC", "staged_state"):
            s = state_of(P_in, prm.get("tK", .5), prm["tC"], name != "late_stateC")
            if name == "staged_state":
                e = pol_gain(P_in, "g7", iE, prm["te"]); keep = e == 0
                mp = learn_map(s[keep], rows[keep], LATE_CAND) if keep.any() else {}
            else: mp = learn_map(s, rows, LATE_CAND)
        a = apply(name, P_in, prm, mp); em = E[rows, a].mean()
        cp = (E[rows, a] - base[rows])[~isCm[rows]].mean()                    # 선택 단계 C+ 안전 제약
        key = (cp >= SEL_CP_MIN, em)
        if best is None or key > best[0]: best = (key, prm, mp)
    return best[1], best[2]

In [6]:
# ===== A1-4. 5-fold 교차 적합 실행 (CPU 약 10~20분) =====
POLS = list(GRID); CHO = {p: np.zeros(N, int) for p in POLS}; SEL = {p: [] for p in POLS}; DIAG = []
t0 = time.time()
for k in range(5):
    tr, te = np.where(fold != k)[0], np.where(fold == k)[0]
    P_in = oof_predict(tr, seed=SEED + k)                                     # 선택용 (test 미사용)
    P_te = fit_predict(tr, te)                                                # 적용용
    DIAG.append(dict(fold=k, AUROC_K7=roc_auc_score(yK[te], P_te["pK7"]), AUROC_K13=roc_auc_score(yK[te], P_te["pK13"]),
                     AUROC_C13=roc_auc_score(yC[te], P_te["pC13"])))
    for p in POLS:
        prm, mp = fit_policy(p, P_in, tr)
        CHO[p][te] = apply(p, P_te, prm, mp)
        SEL[p].append({"fold": k, **prm, **({"map": {str(s): ACT[a] for s, a in mp.items()}} if mp else {})})
    print(f"fold {k} 완료 {time.time() - t0:.0f}s")
DIAG = pd.DataFrame(DIAG); print(DIAG.round(3).to_string(index=False)); DIAG.to_csv(OUT + "A1_probe_diag.csv", index=False)
json.dump(SEL, open(OUT + "A1_selected_params.json", "w"), indent=1, ensure_ascii=False, default=str)

fold 0 완료 20s
fold 1 완료 39s
fold 2 완료 60s
fold 3 완료 78s
fold 4 완료 98s
 fold  AUROC_K7  AUROC_K13  AUROC_C13
    0     0.812      0.799      0.899
    1     0.829      0.835      0.878
    2     0.742      0.730      0.874
    3     0.826      0.830      0.907
    4     0.800      0.805      0.914


In [7]:
# ===== A1-5. 기준·상한 정책 =====
def cross_fit_map(states, cand):
    a = np.zeros(N, int)
    for k in range(5):
        tr, te = fold != k, fold == k
        for s in np.unique(states):
            j = cand[int(np.argmax(E[tr & (states == s)][:, cand].mean(0)))]; a[te & (states == s)] = j
    return a
ALL = np.arange(len(ACT))
BASEPOL = {
    "no-op": np.zeros(N, int),
    "always: late_ret_r0.06 (동결)": np.full(N, REF),
    "always: late_ret_r0.09": np.full(N, ACT.index("late_ret_r0.09")),
    "always: early_ret_r0.06": np.full(N, ACT.index("early_ret_r0.06")),
    "oracle: 정답 C로 late 행동·강도 선택": cross_fit_map(yC, LATE_CAND),
    "oracle: 정답 K×C로 전체 행동 선택": cross_fit_map(CELL, ALL),
    "oracle: 행별 최선 (도달 불가)": E.argmax(1),
}
LABEL = {"early_gain": "early-only (L7, 이득 예측)", "late_gain": "late-only (L13, 이득 예측)",
         "late_stateC": "late-only (L13, 예측 C → 행동·강도)", "late_stateKC": "late-only (L13, 예측 K×C → 행동·강도)",
         "staged_gain": "staged L7→L13 (이득 예측)", "staged_state": "staged L7(이득)→L13(예측 K×C)"}
ROUTE = {LABEL[p]: CHO[p] for p in POLS}

In [8]:
# ===== A1-6. 평가 — A-4 ④ 지표 =====
def ci(v, msk=None):
    msk = np.ones(N, bool) if msk is None else msk
    return np.percentile([v[i][msk[i]].mean() for i in BOOT], [2.5, 97.5])

def evaluate(name, a):
    em = E[np.arange(N), a]; d, dr = em - base, em - E[:, REF]
    b01, b10 = int(((E[:, REF] == 0) & (em == 1)).sum()), int(((E[:, REF] == 1) & (em == 0)).sum())
    lo_p = ci(d, ~isCm)[0]
    return dict(policy=name, EM=em.mean(), dEM=d.mean(),
                dC_minus=d[isCm].mean(), dC_plus=d[~isCm].mean(), dC_plus_lo=lo_p, safe=lo_p > SAFETY,
                d_KpCm=d[CELL == "K+C-"].mean(), d_KmCp=d[CELL == "K-C+"].mean(),
                restored=int(((base == 0) & (em == 1)).sum()), broken=int(((base == 1) & (em == 0)).sum()),
                intervene=(a != 0).mean(), noop=(a == 0).mean(),
                early=np.isin(a, iE).mean(), late=np.isin(a, iL).mean(),
                vs_ref=dr.mean(), vs_ref_lo=ci(dr)[0], vs_ref_hi=ci(dr)[1],
                mcnemar_p=binomtest(b01, b01 + b10).pvalue if b01 + b10 else 1.0)

rows = [evaluate(n, a) for n, a in {**BASEPOL, **ROUTE}.items()]

# 무작위 대조: 각 router의 행동 분포를 fold 안에서 섞음
rng = np.random.RandomState(SEED + 5)
for n, a in ROUTE.items():
    ems = []
    for _ in range(N_PERM):
        b = a.copy()
        for k in range(5):
            i = np.where(fold == k)[0]; b[i] = rng.permutation(a[i])
        ems.append(E[np.arange(N), b].mean())
    r = [x for x in rows if x["policy"] == n][0]; r["random_EM_mean"], r["random_EM_95"] = np.mean(ems), np.percentile(ems, 95)

T = pd.DataFrame(rows); T.to_csv(OUT + "A1_policies.csv", index=False)
pd.set_option("display.width", 300); pd.set_option("display.max_columns", 40)
cols = ["policy", "EM", "dEM", "dC_minus", "dC_plus", "dC_plus_lo", "safe", "d_KpCm", "d_KmCp", "restored", "broken",
        "intervene", "noop", "early", "late", "vs_ref", "vs_ref_lo", "vs_ref_hi", "mcnemar_p"]
print(T[cols].round(4).to_string(index=False))
print("\n[무작위 대조]"); print(T.dropna(subset=["random_EM_mean"])[["policy", "EM", "random_EM_mean", "random_EM_95"]].round(4).to_string(index=False))

                         policy     EM    dEM  dC_minus  dC_plus  dC_plus_lo  safe  d_KpCm  d_KmCp  restored  broken  intervene   noop  early   late  vs_ref  vs_ref_lo  vs_ref_hi  mcnemar_p
                          no-op 0.5310 0.0000    0.0000   0.0000      0.0000  True  0.0000  0.0000         0       0     0.0000 1.0000 0.0000 0.0000 -0.0686    -0.0858    -0.0515     0.0000
    always: late_ret_r0.06 (동결) 0.5997 0.0686    0.1356   0.0016     -0.0163  True  0.1994 -0.0038       102      18     1.0000 0.0000 0.0000 1.0000  0.0000     0.0000     0.0000     1.0000
         always: late_ret_r0.09 0.6005 0.0694    0.1667  -0.0278     -0.0507 False  0.2370 -0.0451       128      43     1.0000 0.0000 0.0000 1.0000  0.0008    -0.0114     0.0131     1.0000
        always: early_ret_r0.06 0.5792 0.0482    0.0931   0.0033     -0.0065  True  0.1272 -0.0038        66       7     1.0000 0.0000 1.0000 0.0000 -0.0204    -0.0343    -0.0074     0.0035
    oracle: 정답 C로 late 행동·강도 선택 0.6111 0.0801    0

In [9]:
# ===== A1-7. 행동 분포 · 판정 · 저장 =====
AD = pd.DataFrame({n: pd.Series(ACT)[a].value_counts(normalize=True) for n, a in ROUTE.items()}).fillna(0).round(3)
AD.to_csv(OUT + "A1_action_distribution.csv"); print("[router별 행동 분포]"); print(AD.to_string())

pd.DataFrame({"rid": np.arange(N), "cell": CELL, "fold": fold,
              **{n: pd.Series(ACT)[a].values for n, a in {**BASEPOL, **ROUTE}.items()}}).to_parquet(OUT + "A1_actions.parquet", index=False)

R = T.set_index("policy")
best = R.loc[list(ROUTE)].query("safe").sort_values("vs_ref", ascending=False)
top = best.index[0] if len(best) else None
verdict = ("안전 기준을 통과한 router 없음" if top is None else
           f"최고 router '{top}': Always 대비 {R.loc[top].vs_ref*100:+.2f}%p "
           f"[{R.loc[top].vs_ref_lo*100:+.2f}, {R.loc[top].vs_ref_hi*100:+.2f}] → "
           + ("Always-retain보다 유의하게 우수" if R.loc[top].vs_ref_lo > 0 else "Always-retain과 유의차 없음"))
oc = R.loc["oracle: 정답 C로 late 행동·강도 선택"]
json.dump({"verdict": verdict,
           "C_oracle_strength_vs_ref": [float(oc.vs_ref), float(oc.vs_ref_lo), float(oc.vs_ref_hi)], "C_oracle_safe": bool(oc.safe),
           "probe_AUROC_mean": DIAG.mean().drop("fold").round(3).to_dict(),
           "K_label": "kside_v3", "input_order": "C→Q (p8s-v2)", "read_layers": [L_EARLY, L_LATE]},
          open(OUT + "A1_summary.json", "w"), indent=1, ensure_ascii=False)
print("\n판정:", verdict)
print(f"C 정답 강도 분리 상한: {oc.vs_ref*100:+.2f}%p [{oc.vs_ref_lo*100:+.2f}, {oc.vs_ref_hi*100:+.2f}] | 안전 {bool(oc.safe)}")
print("저장:", OUT)

[router별 행동 분포]
                 early-only (L7, 이득 예측)  late-only (L13, 이득 예측)  late-only (L13, 예측 C → 행동·강도)  late-only (L13, 예측 K×C → 행동·강도)  staged L7→L13 (이득 예측)  staged L7(이득)→L13(예측 K×C)
base                              0.000                   0.052                          0.000                            0.065                  0.052                      0.065
early_acc_r0.03                   0.106                   0.000                          0.000                            0.000                  0.000                      0.000
early_acc_r0.06                   0.135                   0.000                          0.000                            0.000                  0.000                      0.000
early_ret_r0.03                   0.197                   0.000                          0.000                            0.000                  0.000                      0.000
early_ret_r0.06                   0.562                   0.000                          0.000